In [1]:
import os
from pathlib import Path
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

review = pd.read_csv('docs/feature_review.csv');
review['final_decision'] = review['caleb_decision'].fillna(review['ai_recommendation'])
keep_cols = review[review['final_decision'].str.lower().str.startswith("keep")]["column"].tolist()

print(keep_cols)
print(len(keep_cols))


c:\Users\clc05\Documents\credit-risk-pd-model
['loan_amnt', 'funded_amnt', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_

In [2]:
import duckdb

conn = duckdb.connect('data/loans.duckdb', read_only=True)
col_sql = ", ".join(["r." + c for c in keep_cols])

loans = conn.sql(f"SELECT s.id, s.default_flag, s.issue_year, s.issue_date, {col_sql} FROM loans_scoped s JOIN loans_raw r ON TRY_CAST(r.id AS BIGINT) = s.id WHERE r.application_type = 'Individual'").df()

print (loans.shape)
print(loans.dtypes.value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(545779, 67)
str               63
int64              2
int32              1
datetime64[us]     1
Name: count, dtype: int64


In [3]:
text_cols = ["emp_length", "home_ownership", "verification_status", "purpose", "earliest_cr_line"]
num_cols = [c for c in keep_cols if c not in text_cols]
print(len(num_cols))

nulls_before = loans[num_cols].isna().sum()

loans[num_cols] = loans[num_cols].apply(pd.to_numeric, errors="coerce")
loans['earliest_cr_line'] = pd.to_datetime(loans['earliest_cr_line'], errors="coerce", format="%b-%Y")
print(loans["earliest_cr_line"].isna().sum())

nulls_after = loans[num_cols].isna().sum()
print(nulls_before)
print(nulls_after)
changed_cols = nulls_after[nulls_after > nulls_before].index
print(loans[changed_cols].isna().sum())

print(loans.dtypes.value_counts())
print(loans.memory_usage(deep=True).sum() / 1e9)

58
0
loan_amnt                              0
funded_amnt                            0
annual_inc                             0
dti                                    0
delinq_2yrs                            0
fico_range_low                         0
fico_range_high                        0
inq_last_6mths                         0
mths_since_last_delinq            271969
mths_since_last_record            449571
open_acc                               0
pub_rec                                0
revol_bal                              0
revol_util                           259
total_acc                              0
collections_12_mths_ex_med             0
mths_since_last_major_derog       392239
acc_now_delinq                         0
tot_coll_amt                           0
tot_cur_bal                            0
total_rev_hi_lim                       0
acc_open_past_24mths                   0
avg_cur_bal                            9
bc_open_to_buy                      5387
bc_util    

In [4]:
never_happened_cols = ["mths_since_last_record", "mths_since_recent_bc_dlq", "mths_since_last_major_derog", "mths_since_recent_revol_delinq", "mths_since_last_delinq", "mths_since_recent_inq"]
for c in never_happened_cols:
    loans["ever_" + c] = loans[c].notna().astype(int)
    loans[c] = loans[c].fillna(loans[c].max())
print(loans[never_happened_cols].isna().sum())

mths_since_last_record            0
mths_since_recent_bc_dlq          0
mths_since_last_major_derog       0
mths_since_recent_revol_delinq    0
mths_since_last_delinq            0
mths_since_recent_inq             0
dtype: int64


In [5]:
median_cols = ["num_tl_120dpd_2m", "mo_sin_old_il_acct", "bc_util", "percent_bc_gt_75", "bc_open_to_buy", "mths_since_recent_bc", "revol_util", "pct_tl_nvr_dlq", "avg_cur_bal", "num_rev_accts"]
for c in median_cols:
    loans[c] = loans[c].fillna(loans.loc[loans['issue_year'] <= 2014, c].median())

print(loans.loc[loans['issue_year'] <= 2014, median_cols].median())
print(loans[median_cols].isna().sum())

num_tl_120dpd_2m           0.0
mo_sin_old_il_acct       128.0
bc_util                   68.5
percent_bc_gt_75          50.0
bc_open_to_buy          3531.0
mths_since_recent_bc      14.0
revol_util                56.1
pct_tl_nvr_dlq           100.0
avg_cur_bal             6374.0
num_rev_accts             14.0
dtype: float64
num_tl_120dpd_2m        0
mo_sin_old_il_acct      0
bc_util                 0
percent_bc_gt_75        0
bc_open_to_buy          0
mths_since_recent_bc    0
revol_util              0
pct_tl_nvr_dlq          0
avg_cur_bal             0
num_rev_accts           0
dtype: int64


In [6]:
loans["emp_length_missing"] = loans["emp_length"].isna().astype(int)
loans["emp_length"] = pd.to_numeric(loans["emp_length"].astype("string").replace({"< 1 year": "0", "10+ years": "10"}).str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")
loans["emp_length"] = loans["emp_length"].fillna(loans["emp_length"].median())
loans["home_ownership"] = loans["home_ownership"].replace({"ANY": "RENT"})
loans["purpose"] = loans["purpose"].replace({"wedding": "other", "renewable_energy": "other", "educational": "other"})

print(loans["emp_length"].value_counts().sort_index())
print(loans["emp_length_missing"].mean())
print(loans["home_ownership"].value_counts())
print(loans["purpose"].value_counts())

emp_length
0      44263
1      35976
2      49278
3      44084
4      31683
5      33384
6      59096
7      27097
8      27321
9      20981
10    172616
Name: count, dtype: Int64
0.06149009031127984
home_ownership
MORTGAGE    256100
RENT        232710
OWN          56969
Name: count, dtype: int64
purpose
debt_consolidation    313228
credit_card           136792
home_improvement       31238
other                  28988
major_purchase         10058
medical                 5797
car                     5183
small_business          5070
moving                  3760
vacation                3657
house                   2008
Name: count, dtype: int64
